# Meeting Presentation Plots — April 27 2026

Generates 5 figures for the thesis progress meeting.

**Data sources (already on Drive):**
- Phase 5 pkl: `epr_spectral_phase5/Qwen2.5-Math-7B-Instruct__math500/phase5_results.pkl`
- Phase 4 pkl: `epr_spectral_phase4/Qwen2.5-Math-7B-Instruct__math500/phase4_results.pkl`
- Phase 6 pkl: `epr_spectral_phase6/Mistral-7B-Instruct-v0.2__hotpotqa/phase6_results.pkl`

**Figures produced:**
1. `fig1_individual_traces.png` — H(n) for individual correct vs incorrect samples (EPR annotated)
2. `fig2_avg_psd.png` — Average PSD: correct vs incorrect
3. `fig3_feature_aucs.png` — Feature AUC bar chart (MATH-500 / Qwen-7B T=1.0)
4. `fig4_results_summary.png` — Progression bar chart: EPR paper → our method → MATH-500 spectral
5. `fig5_avg_trajectories.png` — Average H(n) trajectories (correct vs incorrect, mean ± std)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pickle, itertools
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.interpolate import interp1d

matplotlib.rcParams.update({
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

OUT_DIR = '/content/drive/MyDrive/meeting_plots_apr27'
os.makedirs(OUT_DIR, exist_ok=True)

CORRECT_COLOR = '#2196F3'   # blue
WRONG_COLOR   = '#F44336'   # red
ALPHA_BAND    = 0.15

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def save_fig(name):
    path = os.path.join(OUT_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved: {path}')

print('Setup done. Output dir:', OUT_DIR)

In [ ]:
# Load Phase 5 (MATH-500 / Qwen-7B T=1.0) — 90.0% AUC
P5_PATH = '/content/drive/MyDrive/epr_spectral_phase5/Qwen2.5-Math-7B-Instruct__math500/phase5_results.pkl'
p5 = load_pkl(P5_PATH)

labels_p5    = np.array(p5['raw_labels'])
raw_ents_p5  = p5['raw_entropies']          # list of lists — saved by Phase 5
feat_p5      = p5['feat_arrays']
auc_map_p5   = p5['auc_map']
sign_map_p5  = p5['sign_map']
best_p5      = p5['subset_results'][0] if p5['subset_results'] else None

correct_idx = [i for i, y in enumerate(labels_p5) if y == 1]
wrong_idx   = [i for i, y in enumerate(labels_p5) if y == 0]

print(f'Phase5 MATH-500/Qwen-7B T=1.0:')
print(f'  Total: {len(labels_p5)} samples, {len(correct_idx)} correct, {len(wrong_idx)} wrong')
print(f'  Best AUC: {100*best_p5["auc"]:.1f}%  subset: {best_p5["subset"]}' if best_p5 else '  No subset results')

# Load Phase 4 (MATH-500 / Qwen-7B T=1.5) — 96.6% AUC (best overall)
# Phase 4 results pkl has auc_map + subset_results but NOT raw_entropies.
# Raw traces come from inference_cache.pkl (saved during generation).
P4_RES_PATH   = '/content/drive/MyDrive/epr_spectral_phase4/Qwen2.5-Math-7B-Instruct__math500/phase4_results.pkl'
P4_CACHE_PATH = '/content/drive/MyDrive/epr_spectral_phase4/Qwen2.5-Math-7B-Instruct__math500/inference_cache.pkl'
p4       = load_pkl(P4_RES_PATH)
p4_cache = load_pkl(P4_CACHE_PATH)
best_p4  = p4['subset_results'][0] if p4['subset_results'] else None
n_p4     = p4.get('n_samples', 300)

labels_p4, raw_ents_p4 = [], []
for i in range(n_p4):
    v = p4_cache.get(i, {})
    if v.get('done') and v.get('all_entropies') and len(v['all_entropies']) >= 8:
        labels_p4.append(int(v['correct']))
        raw_ents_p4.append(v['all_entropies'])
labels_p4      = np.array(labels_p4)
correct_idx_p4 = [i for i, y in enumerate(labels_p4) if y == 1]
wrong_idx_p4   = [i for i, y in enumerate(labels_p4) if y == 0]

print(f'Phase4 MATH-500/Qwen-7B T=1.5:')
print(f'  Total: {len(labels_p4)} samples, {len(correct_idx_p4)} correct, {len(wrong_idx_p4)} wrong')
print(f'  Best AUC: {100*best_p4["auc"]:.1f}%  subset: {best_p4["subset"]}' if best_p4 else '  No subset')

# Load Phase 6 (HotpotQA / Mistral-7B) — 59.5% AUC
P6_PATH = '/content/drive/MyDrive/epr_spectral_phase6/Mistral-7B-Instruct-v0.2__hotpotqa/phase6_results.pkl'
p6      = load_pkl(P6_PATH)
best_p6 = p6['subset_results'][0] if p6.get('subset_results') else None
print(f'Phase6 HotpotQA/Mistral-7B T=1.0:')
print(f'  Total: {len(p6["raw_labels"])} samples, best AUC: {100*best_p6["auc"]:.1f}%' if best_p6 else '  No subset')

In [ ]:
# ── Figure 1: Individual H(n) traces — 2×3 grid (top=correct, bottom=incorrect) ─

N_SHOW = 3

correct_sorted = sorted(correct_idx, key=lambda i: len(raw_ents_p5[i]), reverse=True)
wrong_sorted   = sorted(wrong_idx,   key=lambda i: len(raw_ents_p5[i]), reverse=True)
show_correct   = correct_sorted[:N_SHOW]
show_wrong     = wrong_sorted[:N_SHOW]

fig, axes = plt.subplots(2, N_SHOW, figsize=(13, 6), sharey=False)
fig.patch.set_facecolor('white')

row_info = [
    (0, show_correct, CORRECT_COLOR, 'CORRECT'),
    (1, show_wrong,   WRONG_COLOR,   'INCORRECT (hallucination)'),
]

for row, samples, color, row_label in row_info:
    for col, i in enumerate(samples):
        ax   = axes[row][col]
        ents = np.array(raw_ents_p5[i])
        epr  = ents.mean()
        xs   = np.arange(len(ents))

        ax.plot(xs, ents, color=color, lw=1.5, alpha=0.85)
        ax.axhline(epr, color='black', ls='--', lw=1.4, alpha=0.7)
        ax.fill_between(xs, epr, ents,
                        where=(ents > epr), color=color, alpha=0.15)
        ax.fill_between(xs, epr, ents,
                        where=(ents <= epr), color=color, alpha=0.07)

        ax.text(0.97, 0.95, f'EPR={epr:.2f}', transform=ax.transAxes,
                ha='right', va='top', fontsize=8.5,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=color, alpha=0.8))

        ax.set_title(f'{row_label} #{col+1}  (n={len(ents)} tokens)',
                     fontsize=9, color=color, fontweight='bold')
        ax.set_xlabel('Token position n', fontsize=8)
        if col == 0:
            ax.set_ylabel('Entropy H(n)', fontsize=8)

# Shared legend at bottom
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color=CORRECT_COLOR, lw=2,   label='H(n) — correct answer'),
    Line2D([0], [0], color=WRONG_COLOR,   lw=2,   label='H(n) — incorrect answer'),
    Line2D([0], [0], color='black', lw=1.4, ls='--', label='EPR = mean H(n)  (DC component)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           fontsize=9, bbox_to_anchor=(0.5, -0.02), frameon=True)

fig.suptitle(
    'H(n) entropy trajectories — MATH-500 / Qwen2.5-Math-7B-Instruct (T=1.0)\n'
    'EPR discards the trajectory shape — spectral analysis recovers it',
    fontsize=11, fontweight='bold'
)
plt.tight_layout(rect=[0, 0.06, 1, 1])
save_fig('fig1_individual_traces.png')
plt.show()

In [ ]:
# ── Figure 2: Average PSD — correct vs incorrect ───────────────────────────────
#
# Interpolate each PSD to a common frequency grid (DC removed).
# Shaded band = mean ± std across samples.

FREQ_GRID = np.linspace(1e-3, 0.5, 200)

def trace_to_psd(ents):
    e    = np.array(ents, dtype=float)
    e_ac = e - e.mean()                     # remove DC (EPR)
    fft  = np.fft.rfft(e_ac)
    psd  = np.abs(fft) ** 2
    freqs = np.fft.rfftfreq(len(e))
    ac_mask = freqs > 0
    if ac_mask.sum() < 3:
        return None
    f_ac = freqs[ac_mask]
    p_ac = psd[ac_mask]
    p_ac = p_ac / (p_ac.sum() + 1e-12)     # normalise to sum=1
    interp = interp1d(f_ac, p_ac, kind='linear', bounds_error=False, fill_value=0.0)
    return interp(FREQ_GRID)

def collect_psds(indices, raw_ents):
    psds = []
    for i in indices:
        p = trace_to_psd(raw_ents[i])
        if p is not None:
            psds.append(p)
    return np.array(psds)

psd_correct = collect_psds(correct_idx, raw_ents_p5)
psd_wrong   = collect_psds(wrong_idx,   raw_ents_p5)

fig, ax = plt.subplots(figsize=(9, 4.5))

for arr, color, label in [
    (psd_correct, CORRECT_COLOR, f'Correct ({len(psd_correct)} samples)'),
    (psd_wrong,   WRONG_COLOR,   f'Incorrect ({len(psd_wrong)} samples)'),
]:
    mean = arr.mean(0)
    std  = arr.std(0)
    ax.plot(FREQ_GRID, mean, color=color, lw=2.2, label=label)
    ax.fill_between(FREQ_GRID, mean - std, mean + std, color=color, alpha=ALPHA_BAND)

# Annotate spectral regions
ax.axvspan(0.0,  0.10, alpha=0.06, color='green', label='Low band (0–0.1)')
ax.axvspan(0.40, 0.50, alpha=0.06, color='purple', label='High band (0.4–0.5)')

ax.set_xlabel('Normalised frequency (cycles / token)', fontsize=11)
ax.set_ylabel('Normalised PSD (DC removed)', fontsize=11)
ax.set_title(
    'Average Power Spectral Density of H(n) — MATH-500 / Qwen2.5-Math-7B-Instruct (T=1.0)\n'
    'DC removed — orthogonal to EPR by construction',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=9)
plt.tight_layout()
save_fig('fig2_avg_psd.png')
plt.show()

In [ ]:
# ── Figure 3: Feature AUC bar chart — MATH-500 / Qwen-7B T=1.0 ────────────────

FEAT_LABELS = {
    'epr':                  'EPR (mean H(n))',
    'trace_length':         'Trace length',
    'spectral_entropy':     'Spectral entropy',
    'low_band_power':       'Low-band power (0–0.1)',
    'high_band_power':      'High-band power (0.4–0.5)',
    'hl_ratio':             'H/L power ratio',
    'dominant_freq':        'Dominant frequency',
    'spectral_centroid':    'Spectral centroid',
    'stft_max_high_power':  'STFT max high power',
    'stft_spectral_entropy':'STFT spectral entropy',
    'rpdi':                 'RPDI (tail/mean ratio)',
    'sw_var_peak':          'SW variance peak',
}

# Colour coding: spectral (FFT) vs time-domain vs baseline
FEAT_TYPE = {
    'epr': 'baseline',
    'trace_length': 'baseline',
    'spectral_entropy': 'spectral',
    'low_band_power': 'spectral',
    'high_band_power': 'spectral',
    'hl_ratio': 'spectral',
    'dominant_freq': 'spectral',
    'spectral_centroid': 'spectral',
    'stft_max_high_power': 'stft',
    'stft_spectral_entropy': 'stft',
    'rpdi': 'time-domain',
    'sw_var_peak': 'time-domain',
}

TYPE_COLOR = {'baseline': '#9E9E9E', 'spectral': '#2196F3', 'stft': '#673AB7', 'time-domain': '#FF9800'}

# Sort by AUC descending
rows = sorted(auc_map_p5.items(), key=lambda x: x[1], reverse=True)
feat_names_sorted = [r[0] for r in rows]
aucs_sorted       = [r[1] * 100 for r in rows]
labels_sorted     = [FEAT_LABELS.get(n, n) for n in feat_names_sorted]
bar_colors        = [TYPE_COLOR[FEAT_TYPE.get(n, 'baseline')] for n in feat_names_sorted]

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(labels_sorted, aucs_sorted, color=bar_colors, height=0.65, edgecolor='white', linewidth=0.5)

# Value labels
for bar, auc in zip(bars, aucs_sorted):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{auc:.1f}%', va='center', ha='left', fontsize=9)

ax.axvline(50, color='black', ls=':', lw=1.2, label='Chance (50%)')
if best_p5:
    ax.axvline(best_p5['auc'] * 100, color='green', ls='--', lw=1.8,
               label=f'Best Nadler fusion: {100*best_p5["auc"]:.1f}%')

# Legend for feature types
legend_patches = [mpatches.Patch(color=c, label=t) for t, c in TYPE_COLOR.items()]
legend_patches.append(mpatches.Patch(color='white', label=''))  # spacer
ax.legend(handles=legend_patches + [
    plt.Line2D([0], [0], color='black', ls=':', lw=1.2, label='Chance'),
    plt.Line2D([0], [0], color='green', ls='--', lw=1.8, label=f'Best Nadler fusion'),
], fontsize=8, loc='lower right')

ax.set_xlim(45, max(aucs_sorted) + 5)
ax.set_xlabel('AUROC (%)', fontsize=11)
ax.set_title(
    'Individual signal AUROCs — MATH-500 / Qwen2.5-Math-7B-Instruct (T=1.0)\n'
    'Nadler fusion selects orthogonal signals and outperforms all individual features',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
save_fig('fig3_feature_aucs.png')
plt.show()

In [ ]:
# ── Figure 4: Results progression ─────────────────────────────────────────────
# Panel A: grouped vertical bars — Factual QA progression
# Panel B: horizontal bars — Reasoning/math tasks (labels readable at any length)

fig, (ax_qa, ax_math) = plt.subplots(1, 2, figsize=(15, 6),
                                      gridspec_kw={'width_ratios': [1, 1.2]})

# ── Panel A: Factual QA ────────────────────────────────────────────────────────
qa_labels     = ['EPR paper', 'Our T=1.0\nbaseline', '4 temps\nfused', '6 views\n(best)']
triviaqa_aucs = [75.4, 79.1, 80.7, 81.5]
webq_aucs     = [68.2, 71.8, 74.7, 76.0]

x = np.arange(len(qa_labels))
w = 0.38
b1 = ax_qa.bar(x - w/2, triviaqa_aucs, w, color='#42A5F5', label='TriviaQA', edgecolor='white', linewidth=0.5)
b2 = ax_qa.bar(x + w/2, webq_aucs,     w, color='#EF9A9A', label='WebQ',     edgecolor='white', linewidth=0.5)

for bar in list(b1) + list(b2):
    ax_qa.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
               f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax_qa.set_xticks(x)
ax_qa.set_xticklabels(qa_labels, fontsize=10)
ax_qa.set_ylim(62, 90)
ax_qa.set_ylabel('AUROC (%)', fontsize=11)
ax_qa.set_title('(A)  Factual QA — Falcon-3-10B\nContribution 1: multi-view Nadler on EPR',
                fontsize=11, fontweight='bold', pad=10)
ax_qa.legend(fontsize=9)
ax_qa.axhline(75.4, color='gray', ls=':', lw=1.2)
ax_qa.text(3.42, 75.8, 'EPR paper\nbaseline', fontsize=7.5, color='gray', ha='right')

# ── Panel B: Reasoning tasks — horizontal bars ────────────────────────────────
math_labels = [
    'EDIS paper  (GSM8K, single-sample)',
    'Our EPR(trace)  (GSM8K, single-sample)',
    'Our Nadler  (trace+EDIS, GSM8K)',
    'Our Spectral Nadler  (MATH-500, T=1.0)',
    'Our Spectral Nadler  (MATH-500, T=1.5)  ★',
    'LOS-Net supervised  (HotpotQA)',
    'Our Spectral Nadler  (HotpotQA)',
]
math_aucs    = [66.2, 66.8, 68.7, 90.0, 96.6, 72.9, 59.5]
math_colors  = [
    '#BDBDBD',   # EDIS paper — grey
    '#FF9800',   # our EPR(trace) — orange
    '#FFB74D',   # our Nadler GSM8K — light orange
    '#66BB6A',   # MATH-500 T=1.0 — green
    '#2E7D32',   # MATH-500 T=1.5 — dark green (best)
    '#BDBDBD',   # LOS-Net — grey (competitor)
    '#EF9A9A',   # HotpotQA — light red (below LOS-Net)
]

ys   = np.arange(len(math_labels))
bars = ax_math.barh(ys, math_aucs, color=math_colors, height=0.65,
                    edgecolor='white', linewidth=0.5)

for bar, auc in zip(bars, math_aucs):
    ax_math.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height()/2,
                 f'{auc:.1f}%', va='center', ha='left', fontsize=9, fontweight='bold')

ax_math.set_yticks(ys)
ax_math.set_yticklabels(math_labels, fontsize=9.5)
ax_math.set_xlim(50, 104)
ax_math.set_xlabel('AUROC (%)', fontsize=11)
ax_math.set_title('(B)  Reasoning tasks — Contribution 2: spectral H(n) features\n'
                  'Strong on math; scope limited to reasoning (not factual QA)',
                  fontsize=11, fontweight='bold', pad=10)
ax_math.axvline(50, color='black', ls=':', lw=1, alpha=0.4)

# Bracket/annotation for MATH-500
ax_math.annotate('', xy=(97, 4.35), xytext=(97, 3.65),
                 arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.8))
ax_math.text(98, 4.0, 'spectral\nfeatures', ha='left', fontsize=8, color='#2E7D32')

# Divider line between GSM8K rows and MATH-500 rows
ax_math.axhline(2.5, color='gray', ls='--', lw=0.8, alpha=0.5)
ax_math.text(51, 2.6, 'GSM8K', fontsize=7.5, color='gray')
ax_math.text(51, 3.6, 'MATH-500', fontsize=7.5, color='gray')
ax_math.text(51, 5.6, 'HotpotQA', fontsize=7.5, color='gray')

fig.suptitle('Thesis progress — two contributions on hallucination detection (Omri Segev, Apr 2026)',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig('fig4_results_summary.png')
plt.show()

In [ ]:
# ── Figure 5: Average H(n) trajectories (correct vs incorrect) ─────────────────
#
# Align all traces to a normalised [0, 1] position axis via interpolation.
# Shade = mean ± 1 std across samples.

N_GRID = 300
T_GRID = np.linspace(0, 1, N_GRID)

def align_trace(ents):
    e      = np.array(ents, dtype=float)
    xs     = np.linspace(0, 1, len(e))
    interp = interp1d(xs, e, kind='linear')
    return interp(T_GRID)

def collect_aligned(indices, raw_ents):
    return np.array([align_trace(raw_ents[i]) for i in indices
                     if len(raw_ents[i]) >= 8])

# T=1.0 (Phase 5) and T=1.5 (Phase 4) side-by-side — shows effect of temperature
arr_c_p5 = collect_aligned(correct_idx,    raw_ents_p5)
arr_w_p5 = collect_aligned(wrong_idx,      raw_ents_p5)
arr_c_p4 = collect_aligned(correct_idx_p4, raw_ents_p4)
arr_w_p4 = collect_aligned(wrong_idx_p4,   raw_ents_p4)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=False)

for ax, arr_c, arr_w, subtitle in [
    (axes[0], arr_c_p5, arr_w_p5, 'T=1.0  (AUC=90.0%)'),
    (axes[1], arr_c_p4, arr_w_p4, 'T=1.5  (AUC=96.6%)'),
]:
    for arr, color, label in [
        (arr_c, CORRECT_COLOR, f'Correct (n={len(arr_c)})'),
        (arr_w, WRONG_COLOR,   f'Incorrect (n={len(arr_w)})'),
    ]:
        mean = arr.mean(0)
        std  = arr.std(0)
        ax.plot(T_GRID, mean, color=color, lw=2.2, label=label)
        ax.fill_between(T_GRID, mean - std, mean + std, color=color, alpha=ALPHA_BAND)

    ax.axhline(arr_c.mean(), color=CORRECT_COLOR, ls=':', lw=1.2, alpha=0.7,
               label=f'EPR correct = {arr_c.mean():.2f}')
    ax.axhline(arr_w.mean(), color=WRONG_COLOR,   ls=':', lw=1.2, alpha=0.7,
               label=f'EPR wrong = {arr_w.mean():.2f}')
    ax.set_xlabel('Normalised position in response', fontsize=10)
    ax.set_ylabel('Token entropy H(n)', fontsize=10)
    ax.set_title(subtitle, fontsize=11)
    ax.legend(fontsize=8)

fig.suptitle(
    'Average H(n) trajectory — MATH-500 / Qwen2.5-Math-7B-Instruct\n'
    'Incorrect answers show higher and more variable entropy throughout the response\n'
    '(mean ± 1 std across all samples)',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
save_fig('fig5_avg_trajectories.png')
plt.show()

In [ ]:
# ── Final summary ───────────────────────────────────────────────────────────────
print('All figures saved to:', OUT_DIR)
print()
print('Figure list:')
for f in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, f)
    size  = os.path.getsize(fpath) / 1024
    print(f'  {f:<40} {size:.0f} KB')
print()
print('Meeting story:')
print('  fig1 + fig5  →  Act 2: H(n) shape motivates spectral analysis')
print('  fig2         →  Act 2: PSD confirms spectral difference')
print('  fig3         →  Act 3: Which features discriminate (MATH-500)')
print('  fig4         →  Act 3+4: Full results progression + scope')